# Week 2.3: How Words Become Numbers That Mean Something
### Plain-English edition — no coding required

> **How does a computer turn words into numbers (vectors) that carry 'meaning'?**

A computer has no idea what *rice* is. But it can notice that **rice** keeps turning up near
**beans**, **lunch** and **tasty**, and never near **laptop**. Do that across enough text and
you can place every word on a map where **things that appear in the same company end up
sitting together**.

Those positions are called **embeddings**, and they are what makes search, memory and
retrieval possible.

In this notebook you build a real embedding space from scratch — out of 29 short sentences —
and then poke at it.

## How this notebook works
Every grey box below is a **step you run**, not code you read.

1. Some steps have **boxes to fill in** (type a word, drag a slider, tick a box).
2. Press the **▶ play button** on the left of the step.
3. Read the answer that appears underneath.

That's it. The computer writes and runs the code for you.

**Curious what it wrote?** Click **Show code** on any step to read it, then click
**Hide code** in the same place to tuck it away again.

> **Red text means something needs fixing** — read it and press play again.
> **Green means you got it right.** No red at all? You're good.

## What you will learn
- **The idea** — meaning from company kept, not from a dictionary
- **The counting table** — the raw material behind Word2Vec and GloVe
- **The vector** — what a word actually looks like as numbers
- **Similarity** — measuring how close two words are
- **Vector arithmetic** — adding and subtracting meaning
- **The map** — squashing many dimensions down to a picture you can look at
- **Sentence search** — the trick that makes Week 2.4 possible
- **Real embeddings** — what changes when the corpus is the whole internet

---
## Part 1: Meaning comes from company

Before any numbers exist, there is just counting: **which words appear near which other words?**
This is the raw material under Word2Vec, GloVe and every embedding model since.

**Our corpus** — 29 short sentences about running, food, code, family, school, sleep and Accra:

```
i love running every morning before work it keeps me healthy
running and swimming are my favourite sport exercise
i eat rice and beans for lunch it is tasty food
python code laptop computer software program write
...
```

In [ ]:
#@title ▶ Step 1 — Show me the company a word keeps { display-mode: "form" }
#@markdown Pick any word from the corpus: running, rice, code, happy, school, sleep, accra, music...
word = "rice" #@param {type:"string"}
#@markdown How many neighbouring words count as "near"?
window = 3 #@param {type:"slider", min:1, max:6, step:1}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]
CORPUS = [
 "i love running every morning before work it keeps me healthy",
 "running and swimming are my favourite sport exercise",
 "exercise keeps the body healthy strong and fit",
 "football match team play sport running goal",
 "swimming running exercise healthy body fit morning",
 "morning run before work then coffee and breakfast",
 "i eat rice and beans for lunch it is tasty food",
 "food rice beans chicken lunch tasty filling meal",
 "cooking food eating lunch with family tasty meal",
 "rice chicken food dinner eating tasty evening meal",
 "coffee breakfast morning food drink tasty",
 "i write code in python on my laptop computer",
 "python code laptop computer software program write",
 "programming code software laptop computer write evening",
 "computer laptop software program code python work",
 "work office project deadline computer email",
 "family friends happy love together joy laughter",
 "happy friends family laughter joy together weekend",
 "sad tired alone unhappy feeling low",
 "friends family together love happy weekend",
 "school study books learning exams reading",
 "study books school learn exams reading notes",
 "school exams study learning books teacher",
 "music song dance listen radio evening",
 "song music dance happy listen weekend",
 "sleep rest night bed tired early",
 "night sleep bed rest early morning tired",
 "accra ghana city home street market",
 "ghana accra market city home people",
]
THEMES = {
 "sport":    ["running","swimming","exercise","sport","football","fit","healthy","body","match","team","play"],
 "food":     ["rice","beans","food","chicken","lunch","tasty","eat","eating","meal","cooking","dinner","coffee","breakfast"],
 "computer": ["code","python","laptop","computer","software","program","programming","write","work","office"],
 "people":   ["family","friends","happy","love","together","joy","laughter","weekend","sad","tired"],
 "school":   ["school","study","books","learning","exams","reading","learn","notes","teacher"],
}
STOP = {"the","a","an","and","my","i","is","are","was","in","for","with","of","to",
        "me","it","on","that","this","am","then","before","every"}
def content_words(text):
    return [w for w in words_of(text) if w not in STOP]
def build_space(dim=24, window=3):
    T = [content_words(c) for c in CORPUS]
    vocab = sorted({w for t in T for w in t}); idx = {w: i for i, w in enumerate(vocab)}
    M = np.zeros((len(vocab), len(vocab)))
    for t in T:
        for i, w in enumerate(t):
            for j in range(max(0, i - window), min(len(t), i + window + 1)):
                if i != j: M[idx[w], idx[t[j]]] += 1.0
    tot = M.sum(); rows = M.sum(1, keepdims=True); cols = M.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.log((M * tot) / (rows * cols + 1e-12) + 1e-12)
    P[~np.isfinite(P)] = 0; P[P < 0] = 0
    U, S, Vt = np.linalg.svd(P, full_matrices=False)
    V = U[:, :dim] * S[:dim]
    V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    flat = U[:, :2] * S[:2]
    return vocab, idx, V, M, flat
def space():
    if "_SPACE" not in globals(): globals()["_SPACE"] = build_space()
    return globals()["_SPACE"]
def vec(word):
    vocab, idx, V, M, flat = space()
    return V[idx[word]] if word in idx else None
def near(word, k=5):
    vocab, idx, V, M, flat = space()
    sims = V.dot(V[idx[word]]); order = np.argsort(-sims)
    return [(vocab[i], float(sims[i])) for i in order if vocab[i] != word][:k]
def sim(a, b):
    va, vb = vec(a), vec(b)
    return None if va is None or vb is None else float(va.dot(vb))
def theme_of(word):
    for t, ws in THEMES.items():
        if word in ws: return t
    return "other"
def sentence_vec(text):
    vocab, idx, V, M, flat = space()
    vs = [V[idx[w]] for w in content_words(text) if w in idx]
    if not vs: return None
    v = np.mean(vs, axis=0)
    nrm = np.linalg.norm(v)
    return v / nrm if nrm > 0 else None
def simbar(score, width=34):
    filled = max(0, min(width, int(round(width * max(0.0, score)))))
    return "[" + "#" * filled + "." * (width - filled) + "]"
def verdict(score):
    if score >= 0.7: return "very close in meaning"
    if score >= 0.4: return "related"
    if score >= 0.15: return "loosely connected"
    return "nothing to do with each other"
def f2(x):
    return format(0.0 if abs(x) < 0.005 else x, ".2f")

w = word.strip().lower()
vocab, idx, V, M, flat = build_space(window=window)
if w not in idx:
    problem("'" + w + "' is not in this little corpus, so there is nothing to count.")
    note("Try one of these: " + ", ".join(vocab[:18]) + " ...")
else:
    counts = M[idx[w]]
    order = np.argsort(-counts)
    hits = [(vocab[i], int(counts[i])) for i in order if counts[i] > 0][:10]
    print("Every time '" + w + "' appeared, I looked " + str(window) + " word(s) left and right")
    print("and wrote down what I saw. Here is the tally:")
    print("")
    for other, c in hits:
        print("   " + other.ljust(14) + " " + "#" * min(c, 30) + " " + str(c))
    print("")
    print("'" + w + "' appeared " + str(int(M[idx[w]].sum())) + " times next to something.")
    print("")
    success("That tally IS the meaning of '" + w + "' as far as the computer is concerned.")
    print("")
    print("Nobody told it what " + w + " is. It only knows the company the word keeps.")
    print("")
    note("Every word in the corpus gets a row like this. " + str(len(vocab)) + " words means a "
         + str(len(vocab)) + " x " + str(len(vocab)) + " table - mostly zeros, far too big to be useful. "
         "Step 2 squashes it down.")

---
## Part 2: From a huge table to a short list of numbers

The tally table has one column per word — far too wide, and almost all zeros. So we **squash**
it: keep the patterns that explain the most, throw away the rest. The maths is called SVD, and
the result is that every word becomes a short list of numbers instead of a huge sparse row.

That short list is the **embedding**. Word2Vec does the same job a different way (by predicting
neighbours); GloVe does it by fitting the counts directly. Same idea, same output shape.

In [ ]:
#@title ▶ Step 2 — Show me a word as actual numbers { display-mode: "form" }
word = "running" #@param {type:"string"}
#@markdown How many numbers should each word get? (real models use 300 to 3,000)
dimensions = 24 #@param {type:"slider", min:2, max:60, step:2}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]
CORPUS = [
 "i love running every morning before work it keeps me healthy",
 "running and swimming are my favourite sport exercise",
 "exercise keeps the body healthy strong and fit",
 "football match team play sport running goal",
 "swimming running exercise healthy body fit morning",
 "morning run before work then coffee and breakfast",
 "i eat rice and beans for lunch it is tasty food",
 "food rice beans chicken lunch tasty filling meal",
 "cooking food eating lunch with family tasty meal",
 "rice chicken food dinner eating tasty evening meal",
 "coffee breakfast morning food drink tasty",
 "i write code in python on my laptop computer",
 "python code laptop computer software program write",
 "programming code software laptop computer write evening",
 "computer laptop software program code python work",
 "work office project deadline computer email",
 "family friends happy love together joy laughter",
 "happy friends family laughter joy together weekend",
 "sad tired alone unhappy feeling low",
 "friends family together love happy weekend",
 "school study books learning exams reading",
 "study books school learn exams reading notes",
 "school exams study learning books teacher",
 "music song dance listen radio evening",
 "song music dance happy listen weekend",
 "sleep rest night bed tired early",
 "night sleep bed rest early morning tired",
 "accra ghana city home street market",
 "ghana accra market city home people",
]
THEMES = {
 "sport":    ["running","swimming","exercise","sport","football","fit","healthy","body","match","team","play"],
 "food":     ["rice","beans","food","chicken","lunch","tasty","eat","eating","meal","cooking","dinner","coffee","breakfast"],
 "computer": ["code","python","laptop","computer","software","program","programming","write","work","office"],
 "people":   ["family","friends","happy","love","together","joy","laughter","weekend","sad","tired"],
 "school":   ["school","study","books","learning","exams","reading","learn","notes","teacher"],
}
STOP = {"the","a","an","and","my","i","is","are","was","in","for","with","of","to",
        "me","it","on","that","this","am","then","before","every"}
def content_words(text):
    return [w for w in words_of(text) if w not in STOP]
def build_space(dim=24, window=3):
    T = [content_words(c) for c in CORPUS]
    vocab = sorted({w for t in T for w in t}); idx = {w: i for i, w in enumerate(vocab)}
    M = np.zeros((len(vocab), len(vocab)))
    for t in T:
        for i, w in enumerate(t):
            for j in range(max(0, i - window), min(len(t), i + window + 1)):
                if i != j: M[idx[w], idx[t[j]]] += 1.0
    tot = M.sum(); rows = M.sum(1, keepdims=True); cols = M.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.log((M * tot) / (rows * cols + 1e-12) + 1e-12)
    P[~np.isfinite(P)] = 0; P[P < 0] = 0
    U, S, Vt = np.linalg.svd(P, full_matrices=False)
    V = U[:, :dim] * S[:dim]
    V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    flat = U[:, :2] * S[:2]
    return vocab, idx, V, M, flat
def space():
    if "_SPACE" not in globals(): globals()["_SPACE"] = build_space()
    return globals()["_SPACE"]
def vec(word):
    vocab, idx, V, M, flat = space()
    return V[idx[word]] if word in idx else None
def near(word, k=5):
    vocab, idx, V, M, flat = space()
    sims = V.dot(V[idx[word]]); order = np.argsort(-sims)
    return [(vocab[i], float(sims[i])) for i in order if vocab[i] != word][:k]
def sim(a, b):
    va, vb = vec(a), vec(b)
    return None if va is None or vb is None else float(va.dot(vb))
def theme_of(word):
    for t, ws in THEMES.items():
        if word in ws: return t
    return "other"
def sentence_vec(text):
    vocab, idx, V, M, flat = space()
    vs = [V[idx[w]] for w in content_words(text) if w in idx]
    if not vs: return None
    v = np.mean(vs, axis=0)
    nrm = np.linalg.norm(v)
    return v / nrm if nrm > 0 else None
def simbar(score, width=34):
    filled = max(0, min(width, int(round(width * max(0.0, score)))))
    return "[" + "#" * filled + "." * (width - filled) + "]"
def verdict(score):
    if score >= 0.7: return "very close in meaning"
    if score >= 0.4: return "related"
    if score >= 0.15: return "loosely connected"
    return "nothing to do with each other"
def f2(x):
    return format(0.0 if abs(x) < 0.005 else x, ".2f")

w = word.strip().lower()
vocab, idx, V, M, flat = build_space(dim=dimensions)
if w not in idx:
    problem("'" + w + "' is not in this corpus.")
    note("Try: " + ", ".join(sorted(THEMES["sport"] + THEMES["food"])[:14]) + " ...")
else:
    v = V[idx[w]]
    print("The word:  " + w)
    print("")
    print("Before squashing: a row of " + str(len(vocab)) + " counts, "
          + str(int((M[idx[w]] == 0).sum())) + " of them zero.")
    print("After squashing:  " + str(len(v)) + " numbers.")
    print("")
    print("Here it is - this is literally what the computer stores for '" + w + "':")
    print("")
    line = "   "
    for i, x in enumerate(v):
        line += format(x, "+.3f") + "  "
        if (i + 1) % 6 == 0: print(line); line = "   "
    if line.strip(): print(line)
    print("")
    print("Those numbers mean nothing one at a time. Number 7 is not 'sportiness'.")
    print("They only mean something COMPARED to another word's numbers - which is")
    print("the whole of Step 3.")
    print("")
    nb = near(w, 3) if w in space()[1] else []
    if dimensions < 8:
        problem("With only " + str(dimensions) + " numbers per word, a lot of detail has been thrown away.")
        note("Drag 'dimensions' up and watch the neighbours in Step 3 get more sensible.")
    else:
        success("Stored in " + str(len(v)) + " numbers instead of " + str(len(vocab)) + " counts.")

---
## Part 3: Who sits nearby?

If the map is any good, a word's nearest neighbours should be words you would expect to find
in the same conversation.

In [ ]:
#@title ▶ Step 3 — Who are this word's neighbours? { display-mode: "form" }
word = "running" #@param {type:"string"}
#@markdown How many neighbours to show:
how_many = 6 #@param {type:"slider", min:1, max:12, step:1}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]
CORPUS = [
 "i love running every morning before work it keeps me healthy",
 "running and swimming are my favourite sport exercise",
 "exercise keeps the body healthy strong and fit",
 "football match team play sport running goal",
 "swimming running exercise healthy body fit morning",
 "morning run before work then coffee and breakfast",
 "i eat rice and beans for lunch it is tasty food",
 "food rice beans chicken lunch tasty filling meal",
 "cooking food eating lunch with family tasty meal",
 "rice chicken food dinner eating tasty evening meal",
 "coffee breakfast morning food drink tasty",
 "i write code in python on my laptop computer",
 "python code laptop computer software program write",
 "programming code software laptop computer write evening",
 "computer laptop software program code python work",
 "work office project deadline computer email",
 "family friends happy love together joy laughter",
 "happy friends family laughter joy together weekend",
 "sad tired alone unhappy feeling low",
 "friends family together love happy weekend",
 "school study books learning exams reading",
 "study books school learn exams reading notes",
 "school exams study learning books teacher",
 "music song dance listen radio evening",
 "song music dance happy listen weekend",
 "sleep rest night bed tired early",
 "night sleep bed rest early morning tired",
 "accra ghana city home street market",
 "ghana accra market city home people",
]
THEMES = {
 "sport":    ["running","swimming","exercise","sport","football","fit","healthy","body","match","team","play"],
 "food":     ["rice","beans","food","chicken","lunch","tasty","eat","eating","meal","cooking","dinner","coffee","breakfast"],
 "computer": ["code","python","laptop","computer","software","program","programming","write","work","office"],
 "people":   ["family","friends","happy","love","together","joy","laughter","weekend","sad","tired"],
 "school":   ["school","study","books","learning","exams","reading","learn","notes","teacher"],
}
STOP = {"the","a","an","and","my","i","is","are","was","in","for","with","of","to",
        "me","it","on","that","this","am","then","before","every"}
def content_words(text):
    return [w for w in words_of(text) if w not in STOP]
def build_space(dim=24, window=3):
    T = [content_words(c) for c in CORPUS]
    vocab = sorted({w for t in T for w in t}); idx = {w: i for i, w in enumerate(vocab)}
    M = np.zeros((len(vocab), len(vocab)))
    for t in T:
        for i, w in enumerate(t):
            for j in range(max(0, i - window), min(len(t), i + window + 1)):
                if i != j: M[idx[w], idx[t[j]]] += 1.0
    tot = M.sum(); rows = M.sum(1, keepdims=True); cols = M.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.log((M * tot) / (rows * cols + 1e-12) + 1e-12)
    P[~np.isfinite(P)] = 0; P[P < 0] = 0
    U, S, Vt = np.linalg.svd(P, full_matrices=False)
    V = U[:, :dim] * S[:dim]
    V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    flat = U[:, :2] * S[:2]
    return vocab, idx, V, M, flat
def space():
    if "_SPACE" not in globals(): globals()["_SPACE"] = build_space()
    return globals()["_SPACE"]
def vec(word):
    vocab, idx, V, M, flat = space()
    return V[idx[word]] if word in idx else None
def near(word, k=5):
    vocab, idx, V, M, flat = space()
    sims = V.dot(V[idx[word]]); order = np.argsort(-sims)
    return [(vocab[i], float(sims[i])) for i in order if vocab[i] != word][:k]
def sim(a, b):
    va, vb = vec(a), vec(b)
    return None if va is None or vb is None else float(va.dot(vb))
def theme_of(word):
    for t, ws in THEMES.items():
        if word in ws: return t
    return "other"
def sentence_vec(text):
    vocab, idx, V, M, flat = space()
    vs = [V[idx[w]] for w in content_words(text) if w in idx]
    if not vs: return None
    v = np.mean(vs, axis=0)
    nrm = np.linalg.norm(v)
    return v / nrm if nrm > 0 else None
def simbar(score, width=34):
    filled = max(0, min(width, int(round(width * max(0.0, score)))))
    return "[" + "#" * filled + "." * (width - filled) + "]"
def verdict(score):
    if score >= 0.7: return "very close in meaning"
    if score >= 0.4: return "related"
    if score >= 0.15: return "loosely connected"
    return "nothing to do with each other"
def f2(x):
    return format(0.0 if abs(x) < 0.005 else x, ".2f")

w = word.strip().lower()
vocab, idx, V, M, flat = space()
if w not in idx:
    problem("'" + w + "' is not in this corpus, so it has no position on the map.")
    note("This is called being OUT OF VOCABULARY, and Step 8 is about why it matters.")
    note("Words you can try: " + ", ".join(vocab[:20]) + " ...")
else:
    print("Nearest words to '" + w + "':")
    print("")
    for other, s in near(w, how_many):
        print("   " + other.ljust(14) + " " + simbar(s) + " " + f2(s))
    print("")
    top = near(w, 1)[0]
    if top[1] > 0.5:
        success("Closest of all: '" + top[0] + "'.")
    else:
        note("Nothing is particularly close to '" + w + "' - it appears too rarely in these "
             "29 sentences to be placed confidently.")
    print("")
    mine = theme_of(w)
    if mine != "other":
        same = sum(1 for o, s in near(w, how_many) if theme_of(o) == mine)
        print("Of the " + str(how_many) + " neighbours shown, " + str(same)
              + " are from the same theme (" + mine + ") - and nobody ever told the")
        print("computer what the themes were. It worked that out from counting alone.")

---
## Part 4: Measuring the distance

"Close" is not a feeling — it is a number between roughly 0 and 1, called **cosine similarity**.
1.0 means the two words are used identically. 0 means they never keep the same company.

In [ ]:
#@title ▶ Step 4 — How close are these two words? { display-mode: "form" }
first_word = "running" #@param {type:"string"}
second_word = "exercise" #@param {type:"string"}
#@markdown Also compare against a word from a completely different world:
control_word = "laptop" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]
CORPUS = [
 "i love running every morning before work it keeps me healthy",
 "running and swimming are my favourite sport exercise",
 "exercise keeps the body healthy strong and fit",
 "football match team play sport running goal",
 "swimming running exercise healthy body fit morning",
 "morning run before work then coffee and breakfast",
 "i eat rice and beans for lunch it is tasty food",
 "food rice beans chicken lunch tasty filling meal",
 "cooking food eating lunch with family tasty meal",
 "rice chicken food dinner eating tasty evening meal",
 "coffee breakfast morning food drink tasty",
 "i write code in python on my laptop computer",
 "python code laptop computer software program write",
 "programming code software laptop computer write evening",
 "computer laptop software program code python work",
 "work office project deadline computer email",
 "family friends happy love together joy laughter",
 "happy friends family laughter joy together weekend",
 "sad tired alone unhappy feeling low",
 "friends family together love happy weekend",
 "school study books learning exams reading",
 "study books school learn exams reading notes",
 "school exams study learning books teacher",
 "music song dance listen radio evening",
 "song music dance happy listen weekend",
 "sleep rest night bed tired early",
 "night sleep bed rest early morning tired",
 "accra ghana city home street market",
 "ghana accra market city home people",
]
THEMES = {
 "sport":    ["running","swimming","exercise","sport","football","fit","healthy","body","match","team","play"],
 "food":     ["rice","beans","food","chicken","lunch","tasty","eat","eating","meal","cooking","dinner","coffee","breakfast"],
 "computer": ["code","python","laptop","computer","software","program","programming","write","work","office"],
 "people":   ["family","friends","happy","love","together","joy","laughter","weekend","sad","tired"],
 "school":   ["school","study","books","learning","exams","reading","learn","notes","teacher"],
}
STOP = {"the","a","an","and","my","i","is","are","was","in","for","with","of","to",
        "me","it","on","that","this","am","then","before","every"}
def content_words(text):
    return [w for w in words_of(text) if w not in STOP]
def build_space(dim=24, window=3):
    T = [content_words(c) for c in CORPUS]
    vocab = sorted({w for t in T for w in t}); idx = {w: i for i, w in enumerate(vocab)}
    M = np.zeros((len(vocab), len(vocab)))
    for t in T:
        for i, w in enumerate(t):
            for j in range(max(0, i - window), min(len(t), i + window + 1)):
                if i != j: M[idx[w], idx[t[j]]] += 1.0
    tot = M.sum(); rows = M.sum(1, keepdims=True); cols = M.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.log((M * tot) / (rows * cols + 1e-12) + 1e-12)
    P[~np.isfinite(P)] = 0; P[P < 0] = 0
    U, S, Vt = np.linalg.svd(P, full_matrices=False)
    V = U[:, :dim] * S[:dim]
    V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    flat = U[:, :2] * S[:2]
    return vocab, idx, V, M, flat
def space():
    if "_SPACE" not in globals(): globals()["_SPACE"] = build_space()
    return globals()["_SPACE"]
def vec(word):
    vocab, idx, V, M, flat = space()
    return V[idx[word]] if word in idx else None
def near(word, k=5):
    vocab, idx, V, M, flat = space()
    sims = V.dot(V[idx[word]]); order = np.argsort(-sims)
    return [(vocab[i], float(sims[i])) for i in order if vocab[i] != word][:k]
def sim(a, b):
    va, vb = vec(a), vec(b)
    return None if va is None or vb is None else float(va.dot(vb))
def theme_of(word):
    for t, ws in THEMES.items():
        if word in ws: return t
    return "other"
def sentence_vec(text):
    vocab, idx, V, M, flat = space()
    vs = [V[idx[w]] for w in content_words(text) if w in idx]
    if not vs: return None
    v = np.mean(vs, axis=0)
    nrm = np.linalg.norm(v)
    return v / nrm if nrm > 0 else None
def simbar(score, width=34):
    filled = max(0, min(width, int(round(width * max(0.0, score)))))
    return "[" + "#" * filled + "." * (width - filled) + "]"
def verdict(score):
    if score >= 0.7: return "very close in meaning"
    if score >= 0.4: return "related"
    if score >= 0.15: return "loosely connected"
    return "nothing to do with each other"
def f2(x):
    return format(0.0 if abs(x) < 0.005 else x, ".2f")

a = first_word.strip().lower(); b = second_word.strip().lower(); c = control_word.strip().lower()
vocab, idx, V, M, flat = space()
missing = [w for w in (a, b, c) if w and w not in idx]
if missing:
    problem("Not in this corpus: " + ", ".join(missing))
    note("Try words such as: " + ", ".join(vocab[:20]) + " ...")
else:
    s1 = sim(a, b); s2 = sim(a, c)
    print(a + "  vs  " + b)
    print("   " + simbar(s1) + " " + f2(s1) + "   " + verdict(s1))
    print("")
    print(a + "  vs  " + c)
    print("   " + simbar(s2) + " " + f2(s2) + "   " + verdict(s2))
    print("")
    if s1 > s2:
        success("'" + a + "' is closer to '" + b + "' than to '" + c + "' - by "
                + format(s1 - s2, ".2f") + ".")
    elif s2 > s1:
        problem("Unexpected: '" + a + "' came out closer to '" + c + "' than to '" + b + "'.")
        print("")
        print("That is a real limitation, not a bug. With only 29 sentences, a word that")
        print("appears once or twice gets placed badly. Real models read billions of")
        print("sentences for exactly this reason.")
    else:
        note("Dead level - both words keep equally little company with '" + a + "'.")
    print("")
    print("This one number is the engine of everything that follows: searching your")
    print("notes, finding the right chunk, remembering the right fact.")

---
## Part 5: Doing arithmetic with meaning

Because words are now numbers, you can **add and subtract them**. The famous example is
*king − man + woman ≈ queen*. Ours is smaller, so the answers are rougher — but the mechanism
is identical.

In [ ]:
#@title ▶ Step 5 — Word arithmetic { display-mode: "form" }
#@markdown Fill in: **A** is to **B** as **C** is to... ?
A = "running" #@param {type:"string"}
B = "sport" #@param {type:"string"}
C = "rice" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]
CORPUS = [
 "i love running every morning before work it keeps me healthy",
 "running and swimming are my favourite sport exercise",
 "exercise keeps the body healthy strong and fit",
 "football match team play sport running goal",
 "swimming running exercise healthy body fit morning",
 "morning run before work then coffee and breakfast",
 "i eat rice and beans for lunch it is tasty food",
 "food rice beans chicken lunch tasty filling meal",
 "cooking food eating lunch with family tasty meal",
 "rice chicken food dinner eating tasty evening meal",
 "coffee breakfast morning food drink tasty",
 "i write code in python on my laptop computer",
 "python code laptop computer software program write",
 "programming code software laptop computer write evening",
 "computer laptop software program code python work",
 "work office project deadline computer email",
 "family friends happy love together joy laughter",
 "happy friends family laughter joy together weekend",
 "sad tired alone unhappy feeling low",
 "friends family together love happy weekend",
 "school study books learning exams reading",
 "study books school learn exams reading notes",
 "school exams study learning books teacher",
 "music song dance listen radio evening",
 "song music dance happy listen weekend",
 "sleep rest night bed tired early",
 "night sleep bed rest early morning tired",
 "accra ghana city home street market",
 "ghana accra market city home people",
]
THEMES = {
 "sport":    ["running","swimming","exercise","sport","football","fit","healthy","body","match","team","play"],
 "food":     ["rice","beans","food","chicken","lunch","tasty","eat","eating","meal","cooking","dinner","coffee","breakfast"],
 "computer": ["code","python","laptop","computer","software","program","programming","write","work","office"],
 "people":   ["family","friends","happy","love","together","joy","laughter","weekend","sad","tired"],
 "school":   ["school","study","books","learning","exams","reading","learn","notes","teacher"],
}
STOP = {"the","a","an","and","my","i","is","are","was","in","for","with","of","to",
        "me","it","on","that","this","am","then","before","every"}
def content_words(text):
    return [w for w in words_of(text) if w not in STOP]
def build_space(dim=24, window=3):
    T = [content_words(c) for c in CORPUS]
    vocab = sorted({w for t in T for w in t}); idx = {w: i for i, w in enumerate(vocab)}
    M = np.zeros((len(vocab), len(vocab)))
    for t in T:
        for i, w in enumerate(t):
            for j in range(max(0, i - window), min(len(t), i + window + 1)):
                if i != j: M[idx[w], idx[t[j]]] += 1.0
    tot = M.sum(); rows = M.sum(1, keepdims=True); cols = M.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.log((M * tot) / (rows * cols + 1e-12) + 1e-12)
    P[~np.isfinite(P)] = 0; P[P < 0] = 0
    U, S, Vt = np.linalg.svd(P, full_matrices=False)
    V = U[:, :dim] * S[:dim]
    V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    flat = U[:, :2] * S[:2]
    return vocab, idx, V, M, flat
def space():
    if "_SPACE" not in globals(): globals()["_SPACE"] = build_space()
    return globals()["_SPACE"]
def vec(word):
    vocab, idx, V, M, flat = space()
    return V[idx[word]] if word in idx else None
def near(word, k=5):
    vocab, idx, V, M, flat = space()
    sims = V.dot(V[idx[word]]); order = np.argsort(-sims)
    return [(vocab[i], float(sims[i])) for i in order if vocab[i] != word][:k]
def sim(a, b):
    va, vb = vec(a), vec(b)
    return None if va is None or vb is None else float(va.dot(vb))
def theme_of(word):
    for t, ws in THEMES.items():
        if word in ws: return t
    return "other"
def sentence_vec(text):
    vocab, idx, V, M, flat = space()
    vs = [V[idx[w]] for w in content_words(text) if w in idx]
    if not vs: return None
    v = np.mean(vs, axis=0)
    nrm = np.linalg.norm(v)
    return v / nrm if nrm > 0 else None
def simbar(score, width=34):
    filled = max(0, min(width, int(round(width * max(0.0, score)))))
    return "[" + "#" * filled + "." * (width - filled) + "]"
def verdict(score):
    if score >= 0.7: return "very close in meaning"
    if score >= 0.4: return "related"
    if score >= 0.15: return "loosely connected"
    return "nothing to do with each other"
def f2(x):
    return format(0.0 if abs(x) < 0.005 else x, ".2f")

a = A.strip().lower(); b = B.strip().lower(); c = C.strip().lower()
vocab, idx, V, M, flat = space()
missing = [w for w in (a, b, c) if w not in idx]
if missing:
    problem("Not in this corpus: " + ", ".join(missing))
    note("Try: " + ", ".join(vocab[:20]) + " ...")
else:
    print(a + " is to " + b + " as " + c + " is to  ... ?")
    print("")
    print("The sum being done:  " + b + " - " + a + " + " + c)
    v = V[idx[b]] - V[idx[a]] + V[idx[c]]
    nrm = np.linalg.norm(v)
    v = v / nrm if nrm > 0 else v
    sims = V.dot(v); order = np.argsort(-sims)
    answers = [(vocab[i], float(sims[i])) for i in order if vocab[i] not in (a, b, c)][:4]
    print("")
    print("Closest words to the result:")
    for w, s in answers:
        print("   " + w.ljust(14) + " " + simbar(s) + " " + f2(s))
    print("")
    best = answers[0]
    if best[1] > 0.55:
        success("Best answer: '" + best[0] + "'.")
    else:
        note("No strong answer - the top score is only " + f2(best[1])
             + ", so treat this one as a shrug.")
    print("")
    print("Does it look right? Sometimes yes, often roughly. With 29 sentences the map")
    print("is coarse. The point is that MEANING CAN BE ADDED UP at all - something you")
    print("simply cannot do with words as letters.")

---
## Part 6: Looking at the map

Each word has 24 numbers, so it lives in 24 dimensions — impossible to picture. So we
**squash it down to 2** and draw it. Squeezing many dimensions into a few is called
*dimensionality reduction*, and it loses detail on purpose, so you can see the shape.

In [ ]:
#@title ▶ Step 6 — Draw the map { display-mode: "form" }
#@markdown Which themes should I plot?
show_sport = True #@param {type:"boolean"}
show_food = True #@param {type:"boolean"}
show_computer = True #@param {type:"boolean"}
show_people = True #@param {type:"boolean"}
show_school = False #@param {type:"boolean"}
#@markdown Put a ring around one word so you can find it:
highlight = "rice" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]
CORPUS = [
 "i love running every morning before work it keeps me healthy",
 "running and swimming are my favourite sport exercise",
 "exercise keeps the body healthy strong and fit",
 "football match team play sport running goal",
 "swimming running exercise healthy body fit morning",
 "morning run before work then coffee and breakfast",
 "i eat rice and beans for lunch it is tasty food",
 "food rice beans chicken lunch tasty filling meal",
 "cooking food eating lunch with family tasty meal",
 "rice chicken food dinner eating tasty evening meal",
 "coffee breakfast morning food drink tasty",
 "i write code in python on my laptop computer",
 "python code laptop computer software program write",
 "programming code software laptop computer write evening",
 "computer laptop software program code python work",
 "work office project deadline computer email",
 "family friends happy love together joy laughter",
 "happy friends family laughter joy together weekend",
 "sad tired alone unhappy feeling low",
 "friends family together love happy weekend",
 "school study books learning exams reading",
 "study books school learn exams reading notes",
 "school exams study learning books teacher",
 "music song dance listen radio evening",
 "song music dance happy listen weekend",
 "sleep rest night bed tired early",
 "night sleep bed rest early morning tired",
 "accra ghana city home street market",
 "ghana accra market city home people",
]
THEMES = {
 "sport":    ["running","swimming","exercise","sport","football","fit","healthy","body","match","team","play"],
 "food":     ["rice","beans","food","chicken","lunch","tasty","eat","eating","meal","cooking","dinner","coffee","breakfast"],
 "computer": ["code","python","laptop","computer","software","program","programming","write","work","office"],
 "people":   ["family","friends","happy","love","together","joy","laughter","weekend","sad","tired"],
 "school":   ["school","study","books","learning","exams","reading","learn","notes","teacher"],
}
STOP = {"the","a","an","and","my","i","is","are","was","in","for","with","of","to",
        "me","it","on","that","this","am","then","before","every"}
def content_words(text):
    return [w for w in words_of(text) if w not in STOP]
def build_space(dim=24, window=3):
    T = [content_words(c) for c in CORPUS]
    vocab = sorted({w for t in T for w in t}); idx = {w: i for i, w in enumerate(vocab)}
    M = np.zeros((len(vocab), len(vocab)))
    for t in T:
        for i, w in enumerate(t):
            for j in range(max(0, i - window), min(len(t), i + window + 1)):
                if i != j: M[idx[w], idx[t[j]]] += 1.0
    tot = M.sum(); rows = M.sum(1, keepdims=True); cols = M.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.log((M * tot) / (rows * cols + 1e-12) + 1e-12)
    P[~np.isfinite(P)] = 0; P[P < 0] = 0
    U, S, Vt = np.linalg.svd(P, full_matrices=False)
    V = U[:, :dim] * S[:dim]
    V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    flat = U[:, :2] * S[:2]
    return vocab, idx, V, M, flat
def space():
    if "_SPACE" not in globals(): globals()["_SPACE"] = build_space()
    return globals()["_SPACE"]
def vec(word):
    vocab, idx, V, M, flat = space()
    return V[idx[word]] if word in idx else None
def near(word, k=5):
    vocab, idx, V, M, flat = space()
    sims = V.dot(V[idx[word]]); order = np.argsort(-sims)
    return [(vocab[i], float(sims[i])) for i in order if vocab[i] != word][:k]
def sim(a, b):
    va, vb = vec(a), vec(b)
    return None if va is None or vb is None else float(va.dot(vb))
def theme_of(word):
    for t, ws in THEMES.items():
        if word in ws: return t
    return "other"
def sentence_vec(text):
    vocab, idx, V, M, flat = space()
    vs = [V[idx[w]] for w in content_words(text) if w in idx]
    if not vs: return None
    v = np.mean(vs, axis=0)
    nrm = np.linalg.norm(v)
    return v / nrm if nrm > 0 else None
def simbar(score, width=34):
    filled = max(0, min(width, int(round(width * max(0.0, score)))))
    return "[" + "#" * filled + "." * (width - filled) + "]"
def verdict(score):
    if score >= 0.7: return "very close in meaning"
    if score >= 0.4: return "related"
    if score >= 0.15: return "loosely connected"
    return "nothing to do with each other"
def f2(x):
    return format(0.0 if abs(x) < 0.005 else x, ".2f")

vocab, idx, V, M, flat = space()
wanted = []
if show_sport: wanted.append("sport")
if show_food: wanted.append("food")
if show_computer: wanted.append("computer")
if show_people: wanted.append("people")
if show_school: wanted.append("school")
if not wanted:
    problem("No themes ticked, so there is nothing to draw.")
    note("Tick at least one box and press play again.")
else:
    COLOURS = {"sport": "#C8283C", "food": "#E07B00", "computer": "#1a6fc4",
               "people": "#1a7f37", "school": "#7b3fa0"}
    fig, ax = plt.subplots(figsize=(9, 6.5))
    plotted = 0
    for t in wanted:
        xs, ys, labs = [], [], []
        for w in THEMES[t]:
            if w in idx:
                xs.append(flat[idx[w], 0]); ys.append(flat[idx[w], 1]); labs.append(w)
        ax.scatter(xs, ys, s=52, color=COLOURS[t], label=t, zorder=3)
        for x, y, l in zip(xs, ys, labs):
            ax.annotate(l, (x, y), fontsize=9, xytext=(5, 4), textcoords="offset points")
        plotted += len(xs)
    h = highlight.strip().lower()
    if h and h in idx:
        ax.scatter([flat[idx[h], 0]], [flat[idx[h], 1]], s=320, facecolors="none",
                   edgecolors="black", linewidths=2, zorder=4)
    ax.set_title("The word map, squashed from 24 dimensions down to 2")
    ax.set_xlabel("direction 1 (whatever explains the most difference)")
    ax.set_ylabel("direction 2")
    ax.axhline(0, color="#dddddd", lw=1); ax.axvline(0, color="#dddddd", lw=1)
    ax.legend(loc="best", fontsize=9)
    plt.tight_layout(); plt.show()
    print("")
    print(str(plotted) + " words plotted.")
    if h and h not in idx:
        problem("Could not ring '" + h + "' - it is not in this corpus.")
    elif h:
        success("'" + h + "' is the one with the black ring round it.")
    print("")
    print("Look at the clusters. Nobody drew those groups - they fell out of counting")
    print("which words appear near which. That is the entire trick.")
    print("")
    note("Two words that look close here might not really be close: you are seeing 2 of "
         "24 dimensions. The picture is a shadow, not the thing itself.")

---
## Part 7: The trick that makes search work

Here is the step everything has been building towards. Add up the vectors of the words in a
whole sentence and you get a vector **for the sentence**. Now you can ask: *which of my notes
is closest to this question?* — and get an answer without matching a single word.

In [ ]:
#@title ▶ Step 7 — Find the note that answers my question { display-mode: "form" }
#@markdown Four notes about you. Edit them if you like.
note_1 = "I go running every morning and I swim at the weekend" #@param {type:"string"}
note_2 = "For lunch I usually eat rice and beans, sometimes chicken" #@param {type:"string"}
note_3 = "I write python code on my laptop in the evenings" #@param {type:"string"}
note_4 = "I live in Accra and I shop at the market near my home" #@param {type:"string"}
#@markdown Now ask something. Try NOT to reuse the same words as the note.
question = "what sort of exercise do you do?" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]
CORPUS = [
 "i love running every morning before work it keeps me healthy",
 "running and swimming are my favourite sport exercise",
 "exercise keeps the body healthy strong and fit",
 "football match team play sport running goal",
 "swimming running exercise healthy body fit morning",
 "morning run before work then coffee and breakfast",
 "i eat rice and beans for lunch it is tasty food",
 "food rice beans chicken lunch tasty filling meal",
 "cooking food eating lunch with family tasty meal",
 "rice chicken food dinner eating tasty evening meal",
 "coffee breakfast morning food drink tasty",
 "i write code in python on my laptop computer",
 "python code laptop computer software program write",
 "programming code software laptop computer write evening",
 "computer laptop software program code python work",
 "work office project deadline computer email",
 "family friends happy love together joy laughter",
 "happy friends family laughter joy together weekend",
 "sad tired alone unhappy feeling low",
 "friends family together love happy weekend",
 "school study books learning exams reading",
 "study books school learn exams reading notes",
 "school exams study learning books teacher",
 "music song dance listen radio evening",
 "song music dance happy listen weekend",
 "sleep rest night bed tired early",
 "night sleep bed rest early morning tired",
 "accra ghana city home street market",
 "ghana accra market city home people",
]
THEMES = {
 "sport":    ["running","swimming","exercise","sport","football","fit","healthy","body","match","team","play"],
 "food":     ["rice","beans","food","chicken","lunch","tasty","eat","eating","meal","cooking","dinner","coffee","breakfast"],
 "computer": ["code","python","laptop","computer","software","program","programming","write","work","office"],
 "people":   ["family","friends","happy","love","together","joy","laughter","weekend","sad","tired"],
 "school":   ["school","study","books","learning","exams","reading","learn","notes","teacher"],
}
STOP = {"the","a","an","and","my","i","is","are","was","in","for","with","of","to",
        "me","it","on","that","this","am","then","before","every"}
def content_words(text):
    return [w for w in words_of(text) if w not in STOP]
def build_space(dim=24, window=3):
    T = [content_words(c) for c in CORPUS]
    vocab = sorted({w for t in T for w in t}); idx = {w: i for i, w in enumerate(vocab)}
    M = np.zeros((len(vocab), len(vocab)))
    for t in T:
        for i, w in enumerate(t):
            for j in range(max(0, i - window), min(len(t), i + window + 1)):
                if i != j: M[idx[w], idx[t[j]]] += 1.0
    tot = M.sum(); rows = M.sum(1, keepdims=True); cols = M.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.log((M * tot) / (rows * cols + 1e-12) + 1e-12)
    P[~np.isfinite(P)] = 0; P[P < 0] = 0
    U, S, Vt = np.linalg.svd(P, full_matrices=False)
    V = U[:, :dim] * S[:dim]
    V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    flat = U[:, :2] * S[:2]
    return vocab, idx, V, M, flat
def space():
    if "_SPACE" not in globals(): globals()["_SPACE"] = build_space()
    return globals()["_SPACE"]
def vec(word):
    vocab, idx, V, M, flat = space()
    return V[idx[word]] if word in idx else None
def near(word, k=5):
    vocab, idx, V, M, flat = space()
    sims = V.dot(V[idx[word]]); order = np.argsort(-sims)
    return [(vocab[i], float(sims[i])) for i in order if vocab[i] != word][:k]
def sim(a, b):
    va, vb = vec(a), vec(b)
    return None if va is None or vb is None else float(va.dot(vb))
def theme_of(word):
    for t, ws in THEMES.items():
        if word in ws: return t
    return "other"
def sentence_vec(text):
    vocab, idx, V, M, flat = space()
    vs = [V[idx[w]] for w in content_words(text) if w in idx]
    if not vs: return None
    v = np.mean(vs, axis=0)
    nrm = np.linalg.norm(v)
    return v / nrm if nrm > 0 else None
def simbar(score, width=34):
    filled = max(0, min(width, int(round(width * max(0.0, score)))))
    return "[" + "#" * filled + "." * (width - filled) + "]"
def verdict(score):
    if score >= 0.7: return "very close in meaning"
    if score >= 0.4: return "related"
    if score >= 0.15: return "loosely connected"
    return "nothing to do with each other"
def f2(x):
    return format(0.0 if abs(x) < 0.005 else x, ".2f")

notes = [note_1, note_2, note_3, note_4]
notes = [t for t in notes if t.strip()]
qv = sentence_vec(question)
if not notes:
    problem("All four notes are empty. Write at least one, then press play again.")
elif qv is None:
    problem("None of the words in your question are in this little corpus, so I cannot place it on the map.")
    note("Try a question using words such as: running, exercise, food, lunch, code, laptop, market, home.")
else:
    print("Your question: " + question)
    print("")
    scored = []
    for i, t in enumerate(notes, 1):
        tv = sentence_vec(t)
        s = float(qv.dot(tv)) if tv is not None else 0.0
        scored.append((s, i, t))
    scored.sort(reverse=True)
    print("How close each note is to the question:")
    print("")
    for s, i, t in scored:
        print("   note " + str(i) + "  " + simbar(s) + " " + f2(s))
        print("          " + t[:68] + ("..." if len(t) > 68 else ""))
    print("")
    best_s, best_i, best_t = scored[0]
    shared = set(content_words(question)) & set(content_words(best_t))
    if best_s < 0.2:
        problem("Nothing matched well - the best score was only " + f2(best_s) + ".")
        note("Either the answer is not in your notes, or the words are outside this tiny corpus.")
    else:
        success("Best match: note " + str(best_i) + " (" + f2(best_s) + ")")
        print("")
        print("   " + best_t)
        print("")
        if not shared:
            print("Notice: that note shares NO words with your question. A plain word")
            print("search would have found nothing at all. The map found it anyway,")
            print("because the words sit near each other on it.")
        else:
            print("Words shared with the question: " + ", ".join(sorted(shared)))
            note("Try rewording the question to avoid those words - it should still find the right note.")
    print("")
    print("That is retrieval. Swap four notes for four thousand chunks of your documents")
    print("and you have Week 2.4.")

---
## Part 8: What changes with a real embedding model

Yours was built from 29 sentences in about a second. A real one — Word2Vec, GloVe,
or a modern embedding API — is built from billions of sentences. Same idea, different scale.

In [ ]:
#@title ▶ Step 8 — Compare mine with a real one { display-mode: "form" }
#@markdown How many words does the model know?
vocabulary_size = 400000 #@param {type:"slider", min:1000, max:1000000, step:1000}
#@markdown How many numbers per word?
dimensions = 300 #@param {type:"slider", min:50, max:3000, step:50}
#@markdown Try a word that is NOT in our 29 sentences:
test_word = "jollof" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]
CORPUS = [
 "i love running every morning before work it keeps me healthy",
 "running and swimming are my favourite sport exercise",
 "exercise keeps the body healthy strong and fit",
 "football match team play sport running goal",
 "swimming running exercise healthy body fit morning",
 "morning run before work then coffee and breakfast",
 "i eat rice and beans for lunch it is tasty food",
 "food rice beans chicken lunch tasty filling meal",
 "cooking food eating lunch with family tasty meal",
 "rice chicken food dinner eating tasty evening meal",
 "coffee breakfast morning food drink tasty",
 "i write code in python on my laptop computer",
 "python code laptop computer software program write",
 "programming code software laptop computer write evening",
 "computer laptop software program code python work",
 "work office project deadline computer email",
 "family friends happy love together joy laughter",
 "happy friends family laughter joy together weekend",
 "sad tired alone unhappy feeling low",
 "friends family together love happy weekend",
 "school study books learning exams reading",
 "study books school learn exams reading notes",
 "school exams study learning books teacher",
 "music song dance listen radio evening",
 "song music dance happy listen weekend",
 "sleep rest night bed tired early",
 "night sleep bed rest early morning tired",
 "accra ghana city home street market",
 "ghana accra market city home people",
]
THEMES = {
 "sport":    ["running","swimming","exercise","sport","football","fit","healthy","body","match","team","play"],
 "food":     ["rice","beans","food","chicken","lunch","tasty","eat","eating","meal","cooking","dinner","coffee","breakfast"],
 "computer": ["code","python","laptop","computer","software","program","programming","write","work","office"],
 "people":   ["family","friends","happy","love","together","joy","laughter","weekend","sad","tired"],
 "school":   ["school","study","books","learning","exams","reading","learn","notes","teacher"],
}
STOP = {"the","a","an","and","my","i","is","are","was","in","for","with","of","to",
        "me","it","on","that","this","am","then","before","every"}
def content_words(text):
    return [w for w in words_of(text) if w not in STOP]
def build_space(dim=24, window=3):
    T = [content_words(c) for c in CORPUS]
    vocab = sorted({w for t in T for w in t}); idx = {w: i for i, w in enumerate(vocab)}
    M = np.zeros((len(vocab), len(vocab)))
    for t in T:
        for i, w in enumerate(t):
            for j in range(max(0, i - window), min(len(t), i + window + 1)):
                if i != j: M[idx[w], idx[t[j]]] += 1.0
    tot = M.sum(); rows = M.sum(1, keepdims=True); cols = M.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.log((M * tot) / (rows * cols + 1e-12) + 1e-12)
    P[~np.isfinite(P)] = 0; P[P < 0] = 0
    U, S, Vt = np.linalg.svd(P, full_matrices=False)
    V = U[:, :dim] * S[:dim]
    V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    flat = U[:, :2] * S[:2]
    return vocab, idx, V, M, flat
def space():
    if "_SPACE" not in globals(): globals()["_SPACE"] = build_space()
    return globals()["_SPACE"]
def vec(word):
    vocab, idx, V, M, flat = space()
    return V[idx[word]] if word in idx else None
def near(word, k=5):
    vocab, idx, V, M, flat = space()
    sims = V.dot(V[idx[word]]); order = np.argsort(-sims)
    return [(vocab[i], float(sims[i])) for i in order if vocab[i] != word][:k]
def sim(a, b):
    va, vb = vec(a), vec(b)
    return None if va is None or vb is None else float(va.dot(vb))
def theme_of(word):
    for t, ws in THEMES.items():
        if word in ws: return t
    return "other"
def sentence_vec(text):
    vocab, idx, V, M, flat = space()
    vs = [V[idx[w]] for w in content_words(text) if w in idx]
    if not vs: return None
    v = np.mean(vs, axis=0)
    nrm = np.linalg.norm(v)
    return v / nrm if nrm > 0 else None
def simbar(score, width=34):
    filled = max(0, min(width, int(round(width * max(0.0, score)))))
    return "[" + "#" * filled + "." * (width - filled) + "]"
def verdict(score):
    if score >= 0.7: return "very close in meaning"
    if score >= 0.4: return "related"
    if score >= 0.15: return "loosely connected"
    return "nothing to do with each other"
def f2(x):
    return format(0.0 if abs(x) < 0.005 else x, ".2f")

vocab, idx, V, M, flat = space()
mine_v = len(vocab); mine_d = 24
big = vocabulary_size * dimensions * 4 / (1024 * 1024)
small = mine_v * mine_d * 4 / 1024
print("                        yours            a real one")
print("   sentences read       29               billions")
print("   words known          " + str(mine_v).ljust(16) + format(vocabulary_size, ","))
print("   numbers per word     " + str(mine_d).ljust(16) + str(dimensions))
print("   size on disk         " + (format(small, ".0f") + " KB").ljust(16)
      + format(big, ".0f") + " MB")
print("   time to build        about a second   days, on many machines")
print("")
w = test_word.strip().lower()
if w in idx:
    note("'" + w + "' IS in our corpus, so this test does not show anything. "
         "Try a word we never used - jollof, bicycle, hospital, kente.")
else:
    problem("'" + w + "' is out of vocabulary: our map has no position for it at all.")
    print("")
    print("Not a low score. No score. The word simply does not exist in this space,")
    print("so nothing involving it can be searched, compared or retrieved.")
    print("")
    print("Three ways real models deal with this:")
    print("   - read enough text that almost every word has been seen")
    print("   - break unknown words into familiar pieces (jol + lof)")
    print("   - fall back to the characters themselves")
print("")
print("What you should take away:")
print("   - you did NOT build a toy version of something mysterious")
print("   - you built the real thing, small: count the company, squash, compare")
print("   - when you call an embedding API, that is what you are paying for -")
print("     the same three moves, done over an enormous amount of text")

---
## What you just did

| Step | In plain words | The real name for it |
|---|---|---|
| 1 | Counted which words appear near which | Co-occurrence matrix |
| 2 | Squashed the table into a few numbers per word | Dimensionality reduction (SVD) |
| 3 | Looked up a word's neighbours | Nearest-neighbour search |
| 4 | Measured how close two words are | Cosine similarity |
| 5 | Added and subtracted words | Vector arithmetic |
| 6 | Drew the 24 dimensions as a picture | 2-D projection |
| 7 | Found the note that answers a question | Semantic search |
| 8 | Compared yours with a real model | Pre-trained embeddings |

### The one sentence to remember
**Meaning is position.** Once a word is a position on a map, "find me something similar"
stops being a language problem and becomes simple arithmetic.

### A limit worth keeping in mind
The map only knows what it read. A word that never appeared has no position — and a word
that appeared twice has a bad one. Everything downstream inherits that.

### Where this is going
**Week 2.4: RAG** puts Steps 1–7 together: chop your documents (2.1), check they fit (2.2),
place them on a map (this notebook), then retrieve only the pieces that answer the question.

### Want to see the code?
Click **Show code** on any step.